# 06 — Entraînement du set-transformer de NOTES masquées sur Colab (GPU)

Tâche pivot : prédire la **note** d'un film masqué (cible = **résidu** `note − μ − b_i − b_u`)
à partir du sac de films notés du user. **Juge = RMSE held-out vs le mur `μ+b_i+b_u` ≈ 0.860.**

**Prérequis (à faire AVANT) :**
1. **Runtime → Modifier le type d'exécution → GPU** (sinon AMP OFF + très lent ; MPS/CPU NaN local, CUDA OK).
2. **Le code doit être poussé sur GitHub** (Jules push — sinon Colab tourne l'ancien code).
3. **Uploader les 3 artefacts sur Drive** dans le dossier `DRIVE_DIR` ci-dessous :
   `data/processed/sequences.parquet` (~54 Mo), `genome.npy` (~46 Mo), `id_maps.pkl` (~1,5 Mo).

Le clone ne contient PAS `data/processed/` (gitignoré) → d'où l'upload Drive.

In [ ]:
# --- Config (à ajuster si besoin) ---
REPO_URL  = 'https://github.com/JulesV19/customer-behavior-jepa'
REPO_DIR  = '/content/customer-behavior-jepa'
DRIVE_DIR = '/content/drive/MyDrive/cb_jepa/processed'   # où sont les 3 artefacts MovieLens
RUN_NAME  = 'ratings_d128'                                # nom du checkpoint sauvé sur Drive

In [ ]:
# GPU présent ?
!nvidia-smi -L
import torch; print('CUDA dispo :', torch.cuda.is_available())

In [ ]:
# Clone (ou pull si déjà cloné)
import os
if os.path.isdir(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}

In [ ]:
# Monte le Drive puis copie les 3 artefacts dans data/processed/
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p {REPO_DIR}/data/processed
!cp "{DRIVE_DIR}/sequences.parquet" {REPO_DIR}/data/processed/
!cp "{DRIVE_DIR}/genome.npy"        {REPO_DIR}/data/processed/
!cp "{DRIVE_DIR}/id_maps.pkl"       {REPO_DIR}/data/processed/
!ls -lh {REPO_DIR}/data/processed/

In [ ]:
# pyarrow au cas où — NE PAS réinstaller torch (Colab a la version CUDA)
!pip install -q pyarrow

## Entraînement
Vérifie les lignes de log au démarrage : `device=cuda` et `AMP ... : ON`.
À chaque epoch, `val RMSE ... vs mur ...` : le gain (entre parenthèses) doit devenir **positif**
et se creuser — c'est le modèle qui bat les biais. Zone visée : 0.86 → ~0.77 (MF).

In [ ]:
import os, sys
os.chdir(REPO_DIR)
sys.path.insert(0, '.')
from src.train_ratings import run

# d=128, 4 couches par défaut. Monter layers/d_model pour un run costaud.
hist = run(epochs=15, batch_size=128, num_workers=2)

In [ ]:
# Sauve le checkpoint + l'historique sur le Drive (disque Colab éphémère)
!cp {REPO_DIR}/data/processed/ratings.pt          "{DRIVE_DIR}/{RUN_NAME}.pt"
!cp {REPO_DIR}/data/processed/history_ratings.json "{DRIVE_DIR}/history_{RUN_NAME}.json"
print('checkpoint sauvé ->', DRIVE_DIR, RUN_NAME)

## Éval complète (juste après le run)
RMSE modèle vs mur `μ+b_i+b_u` (≈ 0.860) sur le held-out test, décomposition cold-start
par popularité du film, et sonde genome du vecteur `[USER]` (le goût encode-t-il le contenu ?).

In [ ]:
from src.evaluate_ratings import full_report

report = full_report(name='ratings.pt', eval_users=8000, probe_users=4000)